# Assignment 4

In [1]:
import networkx as nx
import pandas as pd
import numpy as np
import pickle

---

## Part 1 - Random Graph Identification

For the first part of this assignment you will analyze randomly generated graphs and determine which algorithm created them.

In [2]:
G1 = nx.read_gpickle("assets/A4_P1_G1")
G2 = nx.read_gpickle("assets/A4_P1_G2")
G3 = nx.read_gpickle("assets/A4_P1_G3")
G4 = nx.read_gpickle("assets/A4_P1_G4")
G5 = nx.read_gpickle("assets/A4_P1_G5")
P1_Graphs = [G1, G2, G3, G4, G5]

<br>
`P1_Graphs` is a list containing 5 networkx graphs. Each of these graphs were generated by one of three possible algorithms:
* Preferential Attachment (`'PA'`)
* Small World with low probability of rewiring (`'SW_L'`)
* Small World with high probability of rewiring (`'SW_H'`)

Anaylze each of the 5 graphs using any methodology and determine which of the three algorithms generated each graph.

*The `graph_identification` function should return a list of length 5 where each element in the list is either `'PA'`, `'SW_L'`, or `'SW_H'`.*

In [22]:
def graph_identification():
   


    labels = []

    for G in P1_Graphs:
        degrees = np.array([d for _, d in G.degree()])
        max_deg = degrees.max()
        std_deg = degrees.std()
        avg_clust = nx.average_clustering(G)

        # средняя длина пути (на всякий случай проверка связности)
        if nx.is_connected(G):
            avg_path = nx.average_shortest_path_length(G)
        else:
            avg_path = nx.average_shortest_path_length(
                G.subgraph(max(nx.connected_components(G), key=len))
            )

        # Логика классификации
        if max_deg > 30 and avg_clust < 0.05:
            labels.append('PA')
        elif avg_clust > 0.45 and avg_path > 20:
            labels.append('SW_L')
        else:
            labels.append('SW_H')

    return labels

In [23]:
ans_one = graph_identification()
assert type(ans_one) == list, "You must return a list"


---

## Part 2 - Company Emails

For the second part of this assignment you will be working with a company's email network where each node corresponds to a person at the company, and each edge indicates that at least one email has been sent between two people.

The network also contains the node attributes `Department` and `ManagmentSalary`.

`Department` indicates the department in the company which the person belongs to, and `ManagmentSalary` indicates whether that person is receiving a managment position salary.

In [24]:
G = pickle.load(open('assets/email_prediction_NEW.txt', 'rb'))

print(f"Graph with {len(nx.nodes(G))} nodes and {len(nx.edges(G))} edges")

Graph with 1005 nodes and 16706 edges


### Part 2A - Salary Prediction

Using network `G`, identify the people in the network with missing values for the node attribute `ManagementSalary` and predict whether or not these individuals are receiving a managment position salary.

To accomplish this, you will need to create a matrix of node features of your choice using networkx, train a sklearn classifier on nodes that have `ManagementSalary` data, and predict a probability of the node receiving a managment salary for nodes where `ManagementSalary` is missing.



Your predictions will need to be given as the probability that the corresponding employee is receiving a managment position salary.

The evaluation metric for this assignment is the Area Under the ROC Curve (AUC).

Your grade will be based on the AUC score computed for your classifier. A model which with an AUC of 0.75 or higher will recieve full points.

Using your trained classifier, return a Pandas series of length 252 with the data being the probability of receiving managment salary, and the index being the node id.

    Example:
    
        1       1.0
        2       0.0
        5       0.8
        8       1.0
            ...
        996     0.7
        1000    0.5
        1001    0.0
        Length: 252, dtype: float64

In [25]:
list(G.nodes(data=True))[:5] # print the first 5 nodes

[(0, {'Department': 1, 'ManagementSalary': 0.0}),
 (1, {'Department': 1, 'ManagementSalary': nan}),
 (581, {'Department': 3, 'ManagementSalary': 0.0}),
 (6, {'Department': 25, 'ManagementSalary': 1.0}),
 (65, {'Department': 4, 'ManagementSalary': nan})]

In [40]:
list_of_nan = [i for i in list(G.nodes(data=True)) if (i[1]['ManagementSalary']!=1) & (i[1]['ManagementSalary']!=0)]

In [43]:
list_of_not_nan =  [i for i in list(G.nodes(data=True)) if (i[1]['ManagementSalary']!=0) | (i[1]['ManagementSalary']!=1)]

In [45]:
train_X = [[i[0], i[1]['Department']]for i in list_of_not_nan]
train_Y = [i[1]['ManagementSalary'] for i in list_of_not_nan]
test_X = [[i[0], i[1]['Department']]for i in list_of_nan]


In [65]:
def salary_predictions():
    from sklearn.preprocessing import StandardScaler
    from sklearn.ensemble import RandomForestClassifier
    
    clustering = nx.clustering(G)
    degree_centrality = nx.degree_centrality(G)
    degree = nx.degree(G)
    betweenness_centrality = nx.betweenness_centrality(G)
    
    
    list_of_nan = [i for i in list(G.nodes(data=True)) if (i[1]['ManagementSalary']!=1) & (i[1]['ManagementSalary']!=0)]
    list_of_not_nan =  [i for i in list(G.nodes(data=True)) if (i[1]['ManagementSalary']==0) | (i[1]['ManagementSalary']==1)]
    train_X = [[i[0], i[1]['Department'], \
               clustering[i[0]],\
               degree_centrality[i[0]],\
               degree[i[0]],\
               betweenness_centrality[i[0]]]for i in list_of_not_nan]
    
    train_Y = [i[1]['ManagementSalary'] for i in list_of_not_nan]
    
    test_X = [[i[0], i[1]['Department'], \
               clustering[i[0]],\
               degree_centrality[i[0]],\
               degree[i[0]],\
               betweenness_centrality[i[0]]]for i in list_of_nan]
    
    model = RandomForestClassifier()
    
    model.fit(train_X, train_Y)
    
    preds = model.predict_proba(test_X)
    return pd.Series(preds[:,1], index=[i[0]for i in list_of_nan])

    

In [66]:
ans_salary_preds = salary_predictions()
assert type(ans_salary_preds) == pd.core.series.Series, "You must return a Pandas series"
assert len(ans_salary_preds) == 252, "The series must be of length 252"


### Part 2B - New Connections Prediction

For the last part of this assignment, you will predict future connections between employees of the network. The future connections information has been loaded into the variable `future_connections`. The index is a tuple indicating a pair of nodes that currently do not have a connection, and the `Future Connection` column indicates if an edge between those two nodes will exist in the future, where a value of 1.0 indicates a future connection.

In [67]:
future_connections = pd.read_csv('assets/Future_Connections.csv', index_col=0, converters={0: eval})
future_connections.head(10)

,Future Connection
"(6, 840)",0.0
"(4, 197)",0.0
"(620, 979)",0.0
"(519, 872)",0.0
"(382, 423)",0.0
"(97, 226)",1.0
"(349, 905)",0.0
"(429, 860)",0.0
"(309, 989)",0.0
"(468, 880)",0.0


Using network `G` and `future_connections`, identify the edges in `future_connections` with missing values and predict whether or not these edges will have a future connection.

To accomplish this, you will need to:      
1. Create a matrix of features of your choice for the edges found in `future_connections` using Networkx     
2. Train a sklearn classifier on those edges in `future_connections` that have `Future Connection` data     
3. Predict a probability of the edge being a future connection for those edges in `future_connections` where `Future Connection` is missing.



Your predictions will need to be given as the probability of the corresponding edge being a future connection.

The evaluation metric for this assignment is the Area Under the ROC Curve (AUC).

Your grade will be based on the AUC score computed for your classifier. A model which with an AUC of 0.75 or higher will recieve full points.

Using your trained classifier, return a series of length 122112 with the data being the probability of the edge being a future connection, and the index being the edge as represented by a tuple of nodes.

    Example:
    
        (107, 348)    0.35
        (542, 751)    0.40
        (20, 426)     0.55
        (50, 989)     0.35
                  ...
        (939, 940)    0.15
        (555, 905)    0.35
        (75, 101)     0.65
        Length: 122112, dtype: float64

In [81]:
clustering = nx.clustering(G)
degree_centrality = nx.degree_centrality(G)
degree = nx.degree(G)
betweenness_centrality = nx.betweenness_centrality(G)

indxs = list(future_connections.index)

future_connections_copy = future_connections.copy()

in_clustering = [clustering[i[0]] for i in indxs]
in_degree_centrality = [degree_centrality[i[0]] for i in indxs]
in_degree = [degree[i[0]] for i in indxs]
in_betweenness_centrality = [betweenness_centrality[i[0]] for i in indxs]

out_clustering = [clustering[i[1]] for i in indxs]
out_degree_centrality = [degree_centrality[i[1]] for i in indxs]
out_degree = [degree[i[1]] for i in indxs]
out_betweenness_centrality = [betweenness_centrality[i[1]] for i in indxs]

future_connections_copy['in_clustering'] = in_clustering
future_connections_copy['in_degree_centrality'] = in_degree_centrality
future_connections_copy['in_degree'] = in_degree
future_connections_copy['in_betweenness_centrality'] = in_betweenness_centrality

future_connections_copy['out_clustering'] = out_clustering
future_connections_copy['out_degree_centrality'] = out_degree_centrality
future_connections_copy['out_degree'] = out_degree
future_connections_copy['out_betweenness_centrality'] = out_betweenness_centrality

In [87]:
train_X = future_connections_copy.dropna().drop('Future Connection',axis=1)
train_Y = future_connections_copy.dropna()['Future Connection']

test_X = future_connections_copy[future_connections_copy.isna().any(axis=1)].drop('Future Connection',axis=1)

In [90]:
def new_connections_predictions():
    from sklearn.preprocessing import StandardScaler, LabelEncoder
    from sklearn.ensemble import RandomForestClassifier

    clustering = nx.clustering(G)
    degree_centrality = nx.degree_centrality(G)
    degree = nx.degree(G)
    betweenness_centrality = nx.betweenness_centrality(G)

    indxs = list(future_connections.index)

    future_connections_copy = future_connections.copy()

    in_clustering = [clustering[i[0]] for i in indxs]
    in_degree_centrality = [degree_centrality[i[0]] for i in indxs]
    in_degree = [degree[i[0]] for i in indxs]
    in_betweenness_centrality = [betweenness_centrality[i[0]] for i in indxs]

    out_clustering = [clustering[i[1]] for i in indxs]
    out_degree_centrality = [degree_centrality[i[1]] for i in indxs]
    out_degree = [degree[i[1]] for i in indxs]
    out_betweenness_centrality = [betweenness_centrality[i[1]] for i in indxs]

    future_connections_copy['in_clustering'] = in_clustering
    future_connections_copy['in_degree_centrality'] = in_degree_centrality
    future_connections_copy['in_degree'] = in_degree
    future_connections_copy['in_betweenness_centrality'] = in_betweenness_centrality

    future_connections_copy['out_clustering'] = out_clustering
    future_connections_copy['out_degree_centrality'] = out_degree_centrality
    future_connections_copy['out_degree'] = out_degree
    future_connections_copy['out_betweenness_centrality'] = out_betweenness_centrality
    
    train_X = future_connections_copy.dropna().drop('Future Connection',axis=1)
    train_Y = future_connections_copy.dropna()['Future Connection']

    test_X = future_connections_copy[future_connections_copy.isna().any(axis=1)].drop('Future Connection',axis=1)

    
    
    model = RandomForestClassifier()
    
    model.fit(train_X, train_Y)
    
    preds = model.predict_proba(test_X)
    return pd.Series(preds[:,1], index=future_connections_copy[future_connections_copy.isna().any(axis=1)].index)


In [92]:
ans_prob_preds = new_connections_predictions()
assert type(ans_prob_preds) == pd.core.series.Series, "You must return a Pandas series"
assert len(ans_prob_preds) == 122112, "The series must be of length 122112"
